# Orchestration with GenAI Hub

This notebook demonstrates setting up data masking and content filtering, configuring an orchestration pipeline, and querying multiple LLM models with GenAI Hub.

Installing the required packages

The code imports required libraries, reads credentials from a creds.json file, and sets environment variables for authentication and API access

In [1]:
!pip show sap-ai-sdk-gen

Name: sap-ai-sdk-gen
Version: 6.1.2
Summary: SAP Cloud SDK for AI (Python): generative AI SDK
Home-page: https://www.sap.com/
Author: SAP SE
Author-email: 
License: SAP DEVELOPER LICENSE AGREEMENT
Location: C:\Users\C5384965\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: click, dacite, h11, httpx, langchain, langchain-classic, langchain-community, langchain-openai, openai, overloading, packaging, pydantic, sap-ai-sdk-core
Required-by: 


In [ ]:
import time
import json
import os
from IPython.display import clear_output
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
from ai_api_client_sdk.models.parameter_binding import ParameterBinding
from enum import Enum
 
# Inline credentials
with open('creds.json') as f:
    credCF = json.load(f)

# Set environment variables
def set_environment_vars(credCF):
    env_vars = {
        'AICORE_AUTH_URL': credCF['url'] + '/oauth/token',
        'AICORE_CLIENT_ID': credCF['clientid'],
        'AICORE_CLIENT_SECRET': credCF['clientsecret'],
        'AICORE_BASE_URL': credCF["serviceurls"]["AI_API_URL"] + "/v2",
        'AICORE_RESOURCE_GROUP': "grounding" 
    }

    for key, value in env_vars.items():
        os.environ[key] = value
        print(value)

# Create AI Core client instance
def create_ai_core_client(credCF):
    set_environment_vars(credCF)  # Ensure environment variables are set
    return AICoreV2Client(
        base_url=os.environ['AICORE_BASE_URL'],
        auth_url=os.environ['AICORE_AUTH_URL'],
        client_id=os.environ['AICORE_CLIENT_ID'],
        client_secret=os.environ['AICORE_CLIENT_SECRET'],
        resource_group=os.environ['AICORE_RESOURCE_GROUP']
    )

ai_core_client = create_ai_core_client(credCF)

## Basic Orchestration Pipeline

Now that you have YOUR_DEPOLYMENT_URL, let's walk through a basic orchestration pipeline.

In [ ]:

from gen_ai_hub.orchestration_v2.utils import load_text_file 
# Load the support request file content 
support_request_path = r"C:\Users\C5384965\OneDrive - SAP SE\2026\jan-26\06-01-26\ai-core-orchestration-consumption-opt\support-request.txt" # Specify the correct path to the file 
support_request = load_text_file(support_request_path) 
# Print the content to verify it has been loaded 
print(support_request)

"Subject: Bestellung #1234567890 VerspÃ¤tet - John Johnson Nachricht: Halle, ich schreibe ihnen um mich nach dem Status meiner Bestellung mit der Bestellnr. +1234567890 zu erkundigen. Die Lieferung war eigentlich fÃ¼r gestern geplant, ist bisher jedoch nicht erfolgt. Mein Name ist John Johnson und meine Lieferadresse lautet 125 Cole Meadows Drive Palo Alto, California 94301. Bitte lassen Sie mich per Telefon unter der Nummer +1 505802 2172 wissen, wann ich mit meiner Lieferung rechnen kann. Danke!"


# Step 1: Templating

Explanation of Templating Code

This code defines a template for an AI assistant using orchestration configuration. The `Template` object is set up with system and user messages to guide the assistant’s response behavior. 

Key Components:
- **SystemMessage**: Sets a predefined instruction for the AI assistant. This message typically includes the assistant's role and any specific guidelines it should follow.
- **UserMessage**: Represents the user's input and how it is structured in the conversation.
  
In this revised prompt, only queries are passed to the assistant without any additional context. The AI is expected to respond based solely on the provided input.


In [6]:
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import Template

# Define the sentiment analysis template
template = Template(
    template=[
        SystemMessage(content="""You are a customer support assistant. Analyze the sentiment of the user request provided and return whether the sentiment is positive, neutral, or negative. Also provide a one-line justification for your classification."""
        ),
        UserMessage(content="Please analyze the sentiment of the following support request: {{ ?support_text }}"
        ),
    ],
    defaults=
        {"support_text":"User is unhappy with the latest update and facing usability issues."}
)


# Step 2: Define the LLM -list of models

The LLM class is used to configure and initialize a model for generating text based on specific parameters. In this example, we'll use the list of models to perform the content creation task.

ℹ️Note that virtual deployment of the model is managed automatically by the Orchestration Service, so no additional deployment setup is required on your part.

In [8]:
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails

llm = LLMModelDetails(name="gpt-5-nano", parameters={"max_completion_tokens": 1028})

This configuration initializes the model to use the list of llm models with the latest updates. The model will generate responses up to 256 tokens in length and produce more predictable and focused output due to the low temperature setting.

### Data Masking

The Data Masking Module anonymizes or pseudonymizes personally identifiable information (PII) before it is processed by the LLM module. When data is anonymized, all identifying information is replaced with placeholders (e.g., MASKED_ENTITY), and the original data cannot be recovered, ensuring that no trace of the original information is retained. In contrast, pseudonymized data is substituted with unique placeholders (e.g., MASKED_ENTITY_ID), allowing the original information to be restored if needed. In both cases, the masking module identifies sensitive data and replaces it with appropriate placeholders before further processing.

In [9]:
from gen_ai_hub.orchestration_v2.models.data_masking import MaskingModuleConfig, MaskingProviderConfig, MaskingMethod, DPIStandardEntity, ProfileEntity

data_masking_config = MaskingModuleConfig(
    masking_providers=[MaskingProviderConfig(
        method=MaskingMethod.ANONYMIZATION,
        entities=[
            DPIStandardEntity(type=ProfileEntity.ADDRESS),
            DPIStandardEntity(type=ProfileEntity.EMAIL),
            DPIStandardEntity(type=ProfileEntity.PHONE),
            DPIStandardEntity(type=ProfileEntity.PERSON),
        ]
    )],

)

### Content Filtering

The Content Filtering Module can be configured to filter both the input to the LLM module (input filter) and the output generated by the LLM (output filter). The module uses predefined classification services to detect inappropriate or unwanted content, allowing flexible configuration through customizable thresholds. These thresholds can be set to control the sensitivity of filtering, ensuring that content meets desired standards before it is processed or returned as output.

In [10]:
from gen_ai_hub.orchestration_v2.models.azure_content_filter import AzureContentFilter, AzureThreshold
from gen_ai_hub.orchestration_v2.models.llama_guard_3_filter import LlamaGuard38bFilter
from gen_ai_hub.orchestration_v2.models.content_filtering import FilteringModuleConfig, InputFiltering, OutputFiltering
from gen_ai_hub.orchestration_v2.models.content_filter import ContentFilter, ContentFilterProvider

content_filter_config = FilteringModuleConfig(
    input=InputFiltering(filters=[
        ContentFilter(type=ContentFilterProvider.AZURE, config=AzureContentFilter(hate=AzureThreshold.ALLOW_SAFE,
                                                                                  violence=AzureThreshold.ALLOW_SAFE,
                                                                                  self_harm=AzureThreshold.ALLOW_SAFE,
                                                                                  sexual=AzureThreshold.ALLOW_SAFE)),
        ContentFilter(type=ContentFilterProvider.LLAMA_GUARD_3_8B, config=LlamaGuard38bFilter(hate=True))
        ]),
    output=OutputFiltering(filters=[
        ContentFilter(type=ContentFilterProvider.AZURE, config=AzureContentFilter(hate=AzureThreshold.ALLOW_SAFE,
                                                                                  violence=AzureThreshold.ALLOW_SAFE,
                                                                                  self_harm=AzureThreshold.ALLOW_SAFE,
                                                                                  sexual=AzureThreshold.ALLOW_SAFE)),
        ContentFilter(type=ContentFilterProvider.LLAMA_GUARD_3_8B, config=LlamaGuard38bFilter(hate=True))
    ])

)

### Translation

Translation module can be used to translate text from one language to another. You can use this module to translate input text before it is processed by the LLM module, or to translate the output generated by the LLM module. The translation module uses the SAP Document Translation service to perform the translation.

In [11]:
from gen_ai_hub.orchestration_v2.models.translation import TranslationModuleConfig, SAPDocumentTranslation, TranslationConfig

translation_config = TranslationModuleConfig(
    input=SAPDocumentTranslation(
        config=TranslationConfig(
            source_language="de-DE",
            target_language="en-US"
        )
    ),
    output=SAPDocumentTranslation(
        config=TranslationConfig(
            source_language="en-US",
            target_language="de-DE"
        )
    )
)

# Step 3: Create the Orchestration Configuration

The OrchestrationConfig class is used to create a configuration that integrates various components, such as templates and llm models, into a unified orchestration setup. This configuration specifies how these components work together to achieve the desired workflow.

In [12]:
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)


module_config = ModuleConfig(prompt_templating=prompt_template, filtering=content_filter_config, masking=data_masking_config,
                            translation= translation_config)

config = OrchestrationConfig(modules=module_config)


# Step 4: Run the Orchestration Request

The OrchestrationService class is used to interact with the orchestration service by providing a configuration and invoking its operations. This service handles the execution of workflows defined by the provided configuration and processes inputs accordingly.

In [13]:
from gen_ai_hub.orchestration_v2.service import OrchestrationService

orchestration_service = OrchestrationService()

# Run orchestration with the provided input (for example, candidate resume content)
result = orchestration_service.run(config=config, placeholder_values={"support_text" : support_request})  


In [14]:
# Extract the response content
response = result.final_result.choices[0].message.content
print(response)

Negativ – Die Nachricht vermittelt Frustration und Besorgnis aufgrund einer verspäteten Lieferung und fordert einen sofortigen Status an.
